# Context Engine Testing Notebook

This notebook tests the ContextEngine for ingestion and retrieval using sample files.

In [28]:
from context_engine import ContextEngine
from context_engine.chunkers import get as get_chunker
from context_engine.loaders import get as get_loader
from context_engine.stores import get as get_store
from context_engine.embedders import get as get_embedder
import os

## Setup

Configure the ContextEngine with default settings using HashEmbedder for local testing.

In [29]:
# Define the data directory
DATA_DIR = "/Users/nnandagopal/Desktop/personal_projects/RAG/context_engine/data"
SAMPLE_PDF = os.path.join(DATA_DIR, "coffee_processing.pdf")

print(f"Data directory: {DATA_DIR}")
print(f"Sample PDF: {SAMPLE_PDF}")
print(f"PDF exists: {os.path.exists(SAMPLE_PDF)}")

Data directory: /Users/nnandagopal/Desktop/personal_projects/RAG/context_engine/data
Sample PDF: /Users/nnandagopal/Desktop/personal_projects/RAG/context_engine/data/coffee_processing.pdf
PDF exists: True


## Initialize ContextEngine

Create a ContextEngine instance with ChromaDB store and HashEmbedder for local testing.

In [30]:
# Initialize components
loader = get_loader("auto")
chunker = get_chunker("recursive")
store = get_store("chroma", embedder=get_embedder("hash"))

engine = ContextEngine(loader=loader, chunker=chunker, store=store)
print("ContextEngine initialized successfully")

ContextEngine initialized successfully


## Test 1: Parse Document

Test parsing a PDF document without chunking.

In [31]:
# Test 1: Ingest Document
doc_ids = engine.ingest(SAMPLE_PDF)
print(f"Ingested {len(doc_ids)} documents")
print(f"Document IDs (first 5): {doc_ids[:5]}")

Loading weights: 100%|██████████| 360/360 [00:00<00:00, 16875.09it/s]
[INFO] 2026-05-18 02:04:55,109 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-18 02:04:55,123 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-18 02:04:55,123 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-18 02:04:55,140 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-18 02:04:55,144 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-18 02:04:55,144 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12

Ingested 22 documents
Document IDs (first 5): ['document:0', 'document:1', 'document:2', 'document:3', 'document:4']


In [47]:
# Test retrieval with a coffee processing query
query = "environment?"
results = engine.retrieve(query, k=3)

print(f"Query: {query}")
print(f"Number of results: {len(results)}")

for i, result in enumerate(results):
    print('-'*50)
    print(f"Result {i+1}:")
    print(result.page_content, end='\n\n')
    print(f"Metadata: {result.metadata}")

Query: environment?
Number of results: 3
--------------------------------------------------
Result 1:
requires significant amounts of clean water and generates wastewater that must be properly managed to avoid environmental contamination.

Metadata: {'chunker': 'recursive', 'source': 'document', 'chunk_id': 12, 'distance': 0.7642977237701416}
--------------------------------------------------
Result 2:
Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical processes. Each method requires different amounts of water, time, and labor, which influences both the cost of production and the environmental impact of coffee farming.

Metadata: {'source': 'document', 'chunker': 'recursive', 'chunk_id': 3, 'distance': 0.8196660876274109}
-----------------------------------------

In [45]:
# Test with another query
query2 = "effect of coffe processing on environment"
results2 = engine.retrieve(query2, k=2)

print(f"Query: {query2}")
print(f"Number of results: {len(results2)}")

for i, result in enumerate(results2):
    print('-'*50)
    print(f"Result {i+1}:")
    print(result.page_content, end='\n\n')
    print(f"Metadata: {result.metadata}")

Query: effect of coffe processing on environment
Number of results: 2
--------------------------------------------------
Result 1:
Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical

Metadata: {'source': 'document', 'chunk_id': 1, 'chunker': 'recursive', 'distance': 0.6535325050354004}
--------------------------------------------------
Result 2:
Following the washing stage, the beans are dried either on raised beds or patios until they reach the optimal moisture content of 11-12 percent. This careful drying process can take anywhere from 7 to 14 day